<a href="https://colab.research.google.com/github/NatShed/SOTA_2026/blob/main/SOTA_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №1: Transfer learning и fine-tuning визуального backbone

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
- Сравнить frozen, partial и full fine-tuning ResNet18.
- Построить confusion matrix и проверить corruption shift.


## Теоретическая часть

Предобученный backbone реализует отображение $f_\theta(x)=z$. Для нового набора классов заменяется головка $g_\phi(z)$; сначала оптимизируется только $\phi$, затем часть или все параметры $\theta$. Чем меньше данных, тем выше риск разрушить полезное представление большим learning rate.

![Пример изображений из руководства PyTorch по transfer learning](https://docs.pytorch.org/tutorials/_images/sphx_glr_transfer_learning_tutorial_001.png)

Нужно сравнивать не только accuracy, но и macro-F1, время, память и устойчивость к domain shift.


### Математическая постановка

Для многоклассовой задачи используется cross-entropy:
$$\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i).$$
Три режима эксперимента: frozen backbone, размороженный последний блок, полный fine-tuning.


### Материалы

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision models](https://pytorch.org/vision/stable/models.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
2. Сравнить frozen, partial и full fine-tuning ResNet18.
3. Построить confusion matrix и проверить corruption shift.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** ResNet18, FP32/FP16, 32–224 px.
- **Ограничение данных:** до 10 000 train-изображений в обязательном запуске.
- **Fallback:** CPU/FAST_DEV_RUN на подвыборке.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [ ]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cpu', 'fast_dev_run': False, 'data_root': '/data'}


In [ ]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Разделение CIFAR-10

*Вес: 1 балл.*

**Постановка.** Изоляция test необходима для оценки обобщения без подбора по тестовым меткам.

Выполните следующие действия:
1. Загрузите CIFAR-10 и разделите train на стратифицированные train и validation.
2. Подготовьте DataLoader с детерминированным test-преобразованием.

**Результат.** Непересекающиеся train, validation и test и загрузчики train_loader, val_loader, test_loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
from torchvision import transforms, datasets, models
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix
import torch.nn as nn
import torch.optim as optim
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Гиперпараметры ---
IMG_SIZE     = 224 if DEVICE == "cuda" else 96
BATCH_SIZE   = 64  if DEVICE == "cuda" else 32
NUM_WORKERS  = 2
EPOCHS       = 1 if FAST_DEV_RUN else 5
LR           = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES  = 10

TRAIN_N = 1000 if FAST_DEV_RUN else 8000   # ≤ 10 000 по профилю
VAL_N   = 200  if FAST_DEV_RUN else 2000

# --- Датасеты (без transform, чтобы сначала разделить по индексам) ---
train_full = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=True)
test_raw   = datasets.CIFAR10(root=DATA_ROOT, train=False, download=True)
targets    = np.array(train_full.targets)

# --- Стратифицированное разделение train / val ---
pool_idx = np.arange(TRAIN_N + VAL_N)
train_idx, val_idx = train_test_split(
    pool_idx, test_size=VAL_N, stratify=targets[pool_idx], random_state=SEED
)

# --- Трансформации ---
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2023, 0.1994, 0.2010)

train_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.RandomCrop(IMG_SIZE, padding=IMG_SIZE // 8),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=False, transform=train_tf)
val_ds   = datasets.CIFAR10(root=DATA_ROOT, train=True,  download=False, transform=eval_tf)
test_ds  = datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=eval_tf)

train_loader = DataLoader(Subset(train_ds, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS,
                          pin_memory=(DEVICE == "cuda"))
val_loader   = DataLoader(Subset(val_ds,   val_idx),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS)

CLASSES = train_full.classes   # атрибут экземпляра, а не класса

# --- Контрольные проверки ---
check(len(set(train_idx) & set(val_idx)) == 0, "train и val не пересекаются")
check(len(train_idx) == TRAIN_N, f"train содержит {TRAIN_N} объектов")
check(len(val_idx)   == VAL_N,   f"val содержит {VAL_N} объектов")
check(len(test_ds)   == 10000,   "test = 10000 объектов")

print("Классы:", CLASSES)

✔ train и val не пересекаются
✔ train содержит 8000 объектов
✔ val содержит 2000 объектов
✔ test = 10000 объектов
Классы: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


### Backbone и функции обучения

*Вес: 1 балл.*

**Постановка.** Режимы заморозки изменяют множество обновляемых параметров. Сравнивать их можно лишь при одинаковой функции потерь.

Выполните следующие действия:
1. Создайте ResNet18 с новой десятиклассовой головой.
2. Определите функции обучения и вычисления accuracy, macro-F1, матрицы ошибок.

**Результат.** Функции build_model, train_epoch и evaluate.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
def build_model(mode="full", backbone="resnet18"):
    """Собирает модель и применяет режим заморозки: frozen / partial / full."""
    if backbone == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        head_params = list(model.fc.parameters())
        last_block  = list(model.layer4.parameters())
    elif backbone == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        head_params = list(model.classifier.parameters())
        last_block  = list(model.features[-1].parameters())
    else:
        raise ValueError(backbone)

    # 1) заморозить всё
    for p in model.parameters():
        p.requires_grad = False
    # 2) разморозить головку
    for p in head_params:
        p.requires_grad = True
    # 3) partial / full
    if mode == "partial":
        for p in last_block:
            p.requires_grad = True
    elif mode == "full":
        for p in model.parameters():
            p.requires_grad = True
    elif mode != "frozen":
        raise ValueError(mode)

    return model.to(DEVICE)


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad()
        out  = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += x.size(0)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, return_preds=False):
    """Loss, accuracy, macro-F1, confusion matrix и (опционально) предсказания."""
    model.eval()
    criterion = nn.CrossEntropyLoss()
    all_preds, all_labels = [], []
    total_loss, n = 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE, non_blocking=True)
        out  = model(x)
        loss = criterion(out, y)
        total_loss += loss.item() * x.size(0)
        all_preds.append(out.argmax(1).cpu().numpy())
        all_labels.append(y.cpu().numpy())
        n += x.size(0)
    y_pred = np.concatenate(all_preds)
    y_true = np.concatenate(all_labels)
    res = {
        "loss":             total_loss / n,
        "accuracy":         float((y_pred == y_true).mean()),
        "macro_f1":         float(f1_score(y_true, y_pred, average="macro")),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }
    if return_preds:
        res["y_true"] = y_true
        res["y_pred"] = y_pred
    return res


check(callable(build_model) and callable(train_epoch) and callable(evaluate),
      "build_model / train_epoch / evaluate определены")

✔ build_model / train_epoch / evaluate определены


### Режимы fine-tuning

*Вес: 1 балл.*

**Постановка.** Валидация определяет режим разморозки до первого обращения к test.

Выполните следующие действия:
1. Обучите frozen, partial и full при одинаковом числе эпох.
2. Сохраните веса и метрики validation; выберите режим по macro-F1.

**Результат.** Файлы resnet18_*.pt и metrics.json с измеренными validation-метриками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
results = {}

for mode in ["frozen", "partial", "full"]:
    print(f"\n=== Режим: {mode} ===")
    model = build_model(mode=mode, backbone="resnet18")
    trainable = [p for p in model.parameters() if p.requires_grad]

    optimizer = optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    last_val = None
    for ep in range(EPOCHS):
        tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion)
        last_val = evaluate(model, val_loader)
        scheduler.step()
        print(f"  эпоха {ep+1}/{EPOCHS}: train_loss={tr_loss:.3f} "
              f"train_acc={tr_acc:.3f} val_acc={last_val['accuracy']:.3f} "
              f"val_f1={last_val['macro_f1']:.3f}")
    dt = time.time() - t0

    torch.save(model.state_dict(), ARTIFACTS / f"resnet18_{mode}.pt")
    results[mode] = {
        "val_accuracy":     last_val["accuracy"],
        "val_macro_f1":     last_val["macro_f1"],
        "val_loss":         last_val["loss"],
        "time_sec":         dt,
        "trainable_params": sum(p.numel() for p in trainable),
        "epochs":           EPOCHS,
    }
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

with open(ARTIFACTS / "metrics.json", "w") as f:
    json.dump(results, f, indent=2)

best_mode = max(results, key=lambda m: results[m]["val_macro_f1"])
print(f"\nЛучший режим по macro-F1: {best_mode}")
print(json.dumps(results, indent=2))


=== Режим: frozen ===
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:01<00:00, 35.5MB/s]


  эпоха 1/5: train_loss=1.480 train_acc=0.498 val_acc=0.598 val_f1=0.589
  эпоха 2/5: train_loss=1.129 train_acc=0.611 val_acc=0.624 val_f1=0.617
  эпоха 3/5: train_loss=1.050 train_acc=0.640 val_acc=0.643 val_f1=0.640
  эпоха 4/5: train_loss=1.015 train_acc=0.651 val_acc=0.648 val_f1=0.646
  эпоха 5/5: train_loss=0.999 train_acc=0.662 val_acc=0.657 val_f1=0.654

=== Режим: partial ===
  эпоха 1/5: train_loss=0.915 train_acc=0.690 val_acc=0.778 val_f1=0.776
  эпоха 2/5: train_loss=0.621 train_acc=0.792 val_acc=0.801 val_f1=0.799
  эпоха 3/5: train_loss=0.480 train_acc=0.838 val_acc=0.805 val_f1=0.806
  эпоха 4/5: train_loss=0.381 train_acc=0.867 val_acc=0.846 val_f1=0.845
  эпоха 5/5: train_loss=0.284 train_acc=0.904 val_acc=0.859 val_f1=0.857

=== Режим: full ===
  эпоха 1/5: train_loss=1.204 train_acc=0.602 val_acc=0.686 val_f1=0.673
  эпоха 2/5: train_loss=0.840 train_acc=0.716 val_acc=0.735 val_f1=0.731
  эпоха 3/5: train_loss=0.617 train_acc=0.794 val_acc=0.787 val_f1=0.781
  эпох

### Альтернативная архитектура и отложенный тест

*Вес: 1 балл.*

**Постановка.** Архитектуру выбирают на validation, а test используют единожды после выбора.

Выполните следующие действия:
1. Обучите EfficientNet-B0 при том же бюджете.
2. Сравните validation macro-F1 и вычислите test-метрики выбранной модели.

**Результат.** backbone_comparison.json и test_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
# --- Обучаем EfficientNet-B0 в том же режиме, что и победитель ResNet ---
print(f"Обучаем EfficientNet-B0 в режиме {best_mode}")
eff_model = build_model(mode=best_mode, backbone="efficientnet_b0")
trainable = [p for p in eff_model.parameters() if p.requires_grad]

optimizer = optim.AdamW(trainable, lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

t0 = time.time()
val_eff = None
for ep in range(EPOCHS):
    train_epoch(eff_model, train_loader, optimizer, criterion)
    val_eff = evaluate(eff_model, val_loader)
    scheduler.step()
    print(f"  эпоха {ep+1}/{EPOCHS}: val_acc={val_eff['accuracy']:.3f} "
          f"val_f1={val_eff['macro_f1']:.3f}")
dt_eff = time.time() - t0

torch.save(eff_model.state_dict(), ARTIFACTS / f"efficientnet_b0_{best_mode}.pt")

comparison = {
    "resnet18": {
        "mode":         best_mode,
        "val_macro_f1": results[best_mode]["val_macro_f1"],
        "val_accuracy": results[best_mode]["val_accuracy"],
    },
    "efficientnet_b0": {
        "mode":         best_mode,
        "val_macro_f1": val_eff["macro_f1"],
        "val_accuracy": val_eff["accuracy"],
        "time_sec":     dt_eff,
    },
}
with open(ARTIFACTS / "backbone_comparison.json", "w") as f:
    json.dump(comparison, f, indent=2)

# --- Выбор модели ТОЛЬКО по validation ---
if val_eff["macro_f1"] > results[best_mode]["val_macro_f1"]:
    chosen, model = "efficientnet_b0", eff_model
else:
    chosen = "resnet18"
    model  = build_model(mode=best_mode, backbone="resnet18")
    model.load_state_dict(torch.load(ARTIFACTS / f"resnet18_{best_mode}.pt",
                                     map_location=DEVICE))
    del eff_model

# --- ОДИН раз считаем test ---
test_eval = evaluate(model, test_loader, return_preds=True)
test_metrics = {
    "backbone":         chosen,
    "mode":             best_mode,
    "accuracy":         test_eval["accuracy"],
    "macro_f1":         test_eval["macro_f1"],
    "confusion_matrix": test_eval["confusion_matrix"],
}
with open(ARTIFACTS / "test_metrics.json", "w") as f:
    json.dump(test_metrics, f, indent=2)

np.savez(ARTIFACTS / "test_preds.npz",
         y_true=test_eval["y_true"], y_pred=test_eval["y_pred"])

print(f"\nВыбрана {chosen} ({best_mode}); "
      f"test acc = {test_eval['accuracy']:.4f}, "
      f"macro-F1 = {test_eval['macro_f1']:.4f}")

Обучаем EfficientNet-B0 в режиме full
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 113MB/s] 


  эпоха 1/5: val_acc=0.815 val_f1=0.815
  эпоха 2/5: val_acc=0.838 val_f1=0.838
  эпоха 3/5: val_acc=0.876 val_f1=0.877
  эпоха 4/5: val_acc=0.901 val_f1=0.900
  эпоха 5/5: val_acc=0.907 val_f1=0.906

Выбрана efficientnet_b0 (full); test acc = 0.8974, macro-F1 = 0.8970


### Шумовой сдвиг и ошибки классификации

*Вес: 1 балл.*

**Постановка.** Повреждения пикселей позволяют оценить чувствительность выбранного классификатора.

Выполните следующие действия:
1. Примените шум к отложенным изображениям и измерьте accuracy.
2. Сохраните индексы фактических ошибочных предсказаний.

**Результат.** shift_metrics.json и errors.json с предсказаниями модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
SIGMA_SHIFT = 0.2   # отчётный уровень шума

class NoisyDataset(torch.utils.data.Dataset):
    """Гауссов шум в нормализованном пространстве, воспроизводимый по индексу."""
    def __init__(self, base, sigma):
        self.base, self.sigma = base, sigma
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        if self.sigma > 0:
            g = torch.Generator().manual_seed(SEED + int(i))
            x = torch.clamp(x + self.sigma * torch.randn(x.shape, generator=g),
                            -3.0, 3.0)
        return x, y

noisy_loader = DataLoader(NoisyDataset(test_ds, SIGMA_SHIFT),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS)

noisy_eval = evaluate(model, noisy_loader, return_preds=True)

shift_metrics = {
    "sigma":           SIGMA_SHIFT,
    "clean_accuracy":  test_eval["accuracy"],
    "noisy_accuracy":  noisy_eval["accuracy"],
    "accuracy_drop":   test_eval["accuracy"] - noisy_eval["accuracy"],
}
with open(ARTIFACTS / "shift_metrics.json", "w") as f:
    json.dump(shift_metrics, f, indent=2)

err_idx = np.where(noisy_eval["y_pred"] != noisy_eval["y_true"])[0].tolist()
with open(ARTIFACTS / "errors.json", "w") as f:
    json.dump({
        "sigma":         SIGMA_SHIFT,
        "num_errors":    len(err_idx),
        "error_indices": err_idx,
        "examples": [
            {"idx":  int(i),
             "true": CLASSES[int(noisy_eval["y_true"][i])],
             "pred": CLASSES[int(noisy_eval["y_pred"][i])]}
            for i in err_idx[:20]
        ],
    }, f, indent=2)

print(f"Чистая точность: {test_eval['accuracy']:.4f}")
print(f"При σ={SIGMA_SHIFT}: {noisy_eval['accuracy']:.4f} "
      f"(падение {shift_metrics['accuracy_drop']:.4f})")

Чистая точность: 0.8974
При σ=0.2: 0.3323 (падение 0.5651)


### Профиль устойчивости к шуму

*Вес: 1 балл.*

**Постановка.** Одна интенсивность повреждения не описывает характер деградации.

Выполните следующие действия:
1. На одинаковых test-объектах измерьте accuracy для пяти интенсивностей шума.
2. Сохраните профиль деградации.

**Результат.** sigma_sensitivity.json с измеренной точностью.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
SIGMAS = [0.0, 0.05, 0.1, 0.2, 0.4]

profile = []
for s in SIGMAS:
    if s == 0.0:
        acc = float(test_eval["accuracy"])
    else:
        dl  = DataLoader(NoisyDataset(test_ds, s),
                         batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS)
        acc = float(evaluate(model, dl)["accuracy"])
    profile.append({"sigma": float(s), "accuracy": acc})
    print(f"σ = {s:<5}  accuracy = {acc:.4f}")

accs = [p["accuracy"] for p in profile]
check(all(accs[i] >= accs[i+1] - 1e-6 for i in range(len(accs)-1)),
      "точность монотонно не растёт с σ")
check(profile[0]["accuracy"] >= profile[-1]["accuracy"],
      f"чистая ({profile[0]['accuracy']:.4f}) ≥ шумная ({profile[-1]['accuracy']:.4f})")

with open(ARTIFACTS / "sigma_sensitivity.json", "w") as f:
    json.dump(profile, f, indent=2)

# быстрый график (не обязателен, но приятно)
plt.figure(figsize=(6, 4))
plt.plot([p["sigma"] for p in profile], accs, marker="o")
plt.xlabel("σ шума"); plt.ylabel("accuracy")
plt.title(f"Профиль устойчивости: {chosen} ({best_mode})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS / "sigma_profile.png", dpi=120)
plt.close()

print(f"\nСохранено: {ARTIFACTS / 'sigma_sensitivity.json'}")
print(json.dumps(profile, indent=2))

σ = 0.0    accuracy = 0.8974
σ = 0.05   accuracy = 0.8573
σ = 0.1    accuracy = 0.7790
σ = 0.2    accuracy = 0.3323
σ = 0.4    accuracy = 0.1002
✔ точность монотонно не растёт с σ
✔ чистая (0.8974) ≥ шумная (0.1002)

Сохранено: artifacts/sigma_sensitivity.json
[
  {
    "sigma": 0.0,
    "accuracy": 0.8974
  },
  {
    "sigma": 0.05,
    "accuracy": 0.8573
  },
  {
    "sigma": 0.1,
    "accuracy": 0.779
  },
  {
    "sigma": 0.2,
    "accuracy": 0.3323
  },
  {
    "sigma": 0.4,
    "accuracy": 0.1002
  }
]


### Доля обучаемых параметров

*Вес: 1 балл.*

**Постановка.** Число обновляемых весов характеризует стоимость разных режимов адаптации.

Выполните следующие действия:
1. Посчитайте параметры каждого режима разморозки.
2. Проверьте порядок frozen, partial, full.

**Результат.** param_report.json с числами параметров трёх моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
param_report = {}

for mode in ["frozen", "partial", "full"]:
    m = build_model(mode=mode, backbone="resnet18")
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)

    param_report[mode] = {
        "total":              int(total),
        "trainable":          int(trainable),
        "frozen":             int(total - trainable),
        "trainable_fraction": float(trainable / total),
    }
    del m
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

vals = [param_report[m]["trainable"] for m in ["frozen", "partial", "full"]]
check(vals[0] < vals[1] < vals[2],
      f"порядок: frozen={vals[0]} < partial={vals[1]} < full={vals[2]}")

with open(ARTIFACTS / "param_report.json", "w") as f:
    json.dump(param_report, f, indent=2)

print(json.dumps(param_report, indent=2))

✔ порядок: frozen=5130 < partial=8398858 < full=11181642
{
  "frozen": {
    "total": 11181642,
    "trainable": 5130,
    "frozen": 11176512,
    "trainable_fraction": 0.00045878771650889914
  },
  "partial": {
    "total": 11181642,
    "trainable": 8398858,
    "frozen": 2782784,
    "trainable_fraction": 0.7511292169790448
  },
  "full": {
    "total": 11181642,
    "trainable": 11181642,
    "frozen": 0,
    "trainable_fraction": 1.0
  }
}


### Поклассовые ошибки

*Вес: 1 балл.*

**Постановка.** Агрегатная точность может скрыть систематические ошибки отдельных категорий.

Выполните следующие действия:
1. Вычислите recall для каждого класса по test confusion matrix.
2. Найдите класс с наименьшим recall.

**Результат.** class_report.json с поклассовым recall и худшим классом.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
cm = np.array(test_metrics["confusion_matrix"], dtype=float)
recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)

per_class = {CLASSES[i]: float(recall[i]) for i in range(len(CLASSES))}
worst     = CLASSES[int(np.argmin(recall))]
best      = CLASSES[int(np.argmax(recall))]

class_report = {
    "per_class_recall": per_class,
    "worst_class":      worst,
    "worst_recall":     float(recall.min()),
    "best_class":       best,
    "best_recall":      float(recall.max()),
}
with open(ARTIFACTS / "class_report.json", "w") as f:
    json.dump(class_report, f, indent=2)

print(f"Худший класс: {worst}  (recall = {class_report['worst_recall']:.3f})")
print(f"Лучший класс: {best}  (recall = {class_report['best_recall']:.3f})")
for name, r in sorted(per_class.items(), key=lambda kv: kv[1]):
    print(f"  {name:<12} recall = {r:.3f}")

Худший класс: cat  (recall = 0.727)
Лучший класс: truck  (recall = 0.952)
  cat          recall = 0.727
  bird         recall = 0.871
  deer         recall = 0.874
  dog          recall = 0.874
  airplane     recall = 0.923
  frog         recall = 0.928
  horse        recall = 0.937
  ship         recall = 0.939
  automobile   recall = 0.949
  truck        recall = 0.952


### Точность и стоимость обучения

*Вес: 1 балл.*

**Постановка.** Выбор режима зависит не только от качества, но и от длительности обучения.

Выполните следующие действия:
1. Свяжите измеренные время, validation accuracy и количество обучаемых весов.
2. Сохраните сопоставимую таблицу режимов.

**Результат.** efficiency_report.json с наблюдаемыми величинами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
efficiency = {}
for mode in ["frozen", "partial", "full"]:
    t   = max(results[mode]["time_sec"], 1e-6)
    acc = results[mode]["val_accuracy"]
    efficiency[mode] = {
        "time_sec":         results[mode]["time_sec"],
        "val_accuracy":     acc,
        "val_macro_f1":     results[mode]["val_macro_f1"],
        "trainable_params": param_report[mode]["trainable"],
        "acc_per_sec":      acc / t,
        "sec_per_1pct_acc": t / (acc * 100 + 1e-9),
    }

with open(ARTIFACTS / "efficiency_report.json", "w") as f:
    json.dump(efficiency, f, indent=2)

print(f"{'mode':<8}{'time,s':>10}{'val_acc':>10}{'val_f1':>10}"
      f"{'params':>12}{'acc/s':>10}")
for m in ["frozen", "partial", "full"]:
    e = efficiency[m]
    print(f"{m:<8}{e['time_sec']:>10.1f}{e['val_accuracy']:>10.4f}"
          f"{e['val_macro_f1']:>10.4f}{e['trainable_params']:>12d}"
          f"{e['acc_per_sec']:>10.4f}")

mode        time,s   val_acc    val_f1      params     acc/s
frozen       967.7    0.6570    0.6539        5130    0.0007
partial     1515.7    0.8585    0.8571     8398858    0.0006
full        2508.2    0.8660    0.8654    11181642    0.0003


Последний блок подстроился и точность возросла на +0,2. Переучивали все = прирост небольшой, но обучение почти в 2 раза дольше.

### Анализ ошибок при максимальном сдвиге

*Вес: 1 балл.*

**Постановка.** Изменение предсказаний при повреждении полезнее одного агрегированного числа.

Выполните следующие действия:
1. Для выбранной модели получите предсказания на чистом и зашумлённом test для фиксированных индексов.
2. Найдите и сохраните случаи, где правильное чистое предсказание становится ошибочным.

**Результат.** shift_failures.json с индексами и предсказаниями на реальных изображениях.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
MAX_SIGMA = max(SIGMAS)   # 0.4

dl = DataLoader(NoisyDataset(test_ds, MAX_SIGMA),
                batch_size=BATCH_SIZE, shuffle=False,
                num_workers=NUM_WORKERS)
max_eval = evaluate(model, dl, return_preds=True)

y_true      = test_eval["y_true"]
clean_pred  = test_eval["y_pred"]
noisy_pred  = max_eval["y_pred"]

mask    = (clean_pred == y_true) & (noisy_pred != y_true)
fail_idx = np.where(mask)[0].tolist()

failures = [{
    "index":      int(i),
    "true":       CLASSES[int(y_true[i])],
    "clean_pred": CLASSES[int(clean_pred[i])],
    "noisy_pred": CLASSES[int(noisy_pred[i])],
} for i in fail_idx[:100]]

shift_failures = {
    "sigma":             MAX_SIGMA,
    "total_regressions": len(fail_idx),
    "examples":          failures,
}
with open(ARTIFACTS / "shift_failures.json", "w") as f:
    json.dump(shift_failures, f, indent=2)

print(f"При σ={MAX_SIGMA} сломалось {len(fail_idx)} ранее верных предсказаний")
for ex in failures[:5]:
    print(f"  idx={ex['index']:>5}  {ex['true']:<12} "
          f"{ex['clean_pred']:<12} → {ex['noisy_pred']}")

При σ=0.4 сломалось 8049 ранее верных предсказаний
  idx=    1  ship         ship         → airplane
  idx=    4  frog         frog         → airplane
  idx=    5  frog         frog         → airplane
  idx=    6  automobile   automobile   → airplane
  idx=    7  frog         frog         → airplane


### Творческий бонус: новый тип сдвига

*Вес: 1 дополнительный балл.*

**Постановка.** Не все повреждения изображений похожи на гауссовский шум.

Выполните следующие действия:
1. Выберите воспроизводимое фотометрическое или геометрическое преобразование test-изображений.
2. Сравните точность с чистым test и приведите три конкретных примера изменения предсказания.

**Результат.** bonus_shift.json с описанием преобразования, метрикой и индексами примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [ ]:
class BrightnessDataset(torch.utils.data.Dataset):
    """Умножает нормализованный тензор на коэффициент (яркость)."""
    def __init__(self, base, factor):
        self.base, self.factor = base, factor
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        x, y = self.base[i]
        return torch.clamp(x * self.factor, -3.0, 3.0), y

bonus_results = {"clean_accuracy": test_eval["accuracy"], "per_factor": {}}
for factor in [0.5, 0.75, 1.25, 1.5]:
    dl  = DataLoader(BrightnessDataset(test_ds, factor),
                     batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=NUM_WORKERS)
    acc = float(evaluate(model, dl)["accuracy"])
    bonus_results["per_factor"][f"factor_{factor}"] = acc
    print(f"brightness × {factor}: acc = {acc:.4f}")

# три примера смены предсказания при factor=0.5
dl = DataLoader(BrightnessDataset(test_ds, 0.5),
                batch_size=BATCH_SIZE, shuffle=False,
                num_workers=NUM_WORKERS)
beval = evaluate(model, dl, return_preds=True)

mask = (test_eval["y_pred"] == y_true) & (beval["y_pred"] != y_true)
idx  = np.where(mask)[0][:3]
examples = [{
    "index":      int(i),
    "true":       CLASSES[int(y_true[i])],
    "clean_pred": CLASSES[int(test_eval["y_pred"][i])],
    "dim_pred":   CLASSES[int(beval["y_pred"][i])],
} for i in idx]

bonus_shift = {
    "transform":      "brightness scaling (умножение нормализованного тензора)",
    "clean_accuracy": test_eval["accuracy"],
    "results":        bonus_results["per_factor"],
    "examples":       examples,
}
with open(ARTIFACTS / "bonus_shift.json", "w") as f:
    json.dump(bonus_shift, f, indent=2)

print("\nПримеры при затемнении ×0.5:")
for ex in examples:
    print(f"  idx={ex['index']:>5}  true={ex['true']:<12} "
          f"{ex['clean_pred']:<12} → {ex['dim_pred']}")

brightness × 0.5: acc = 0.8739
brightness × 0.75: acc = 0.8991
brightness × 1.25: acc = 0.8928
brightness × 1.5: acc = 0.8807

Примеры при затемнении ×0.5:
  idx=   10  true=airplane     airplane     → cat
  idx=   24  true=dog          dog          → deer
  idx=   35  true=bird         bird         → cat


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


На отложенном test CIFAR-10 (10 000 изображений, использован один раз после выбора по validation) лучшая модель EfficientNet-B0 в режиме full дала accuracy 0,8974 и macro-F1 0,8970; среди ResNet18 full показал 0,866/0,865, partial — 0,859/0,857, frozen — 0,657/0,654, то есть основной выигрыш дала разморозка последнего блока, а полный fine-tuning добавил лишь +0,7 п.п. при росте времени с 1516 до 2508 с (frozen — 968 с). По ошибкам хуже всего распознаётся cat (recall 0,727), лучше всего truck (0,952); гауссов шум σ=0,2 снижает accuracy до 0,3323 (падение 0,5651), σ=0,4 — до 0,1002, все запихивается в класс airplane, тогда как затемнение ×0,5 даёт лишь 0,8739.

## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.


1. Лучше всего partial/full, но full отнимает больше времени, при этом прирост не такой большой, и не стоит затрат, поэтому разморозить только последний блок - самый выгодный шаг. Переход от frozen к partial даёт скачок с 0,654 до 0,857 macro-F1 (+20 п.п.), а весов становится больше «всего» в 1,6 тыс. раз. Это главный вывод: последний блок несёт смысл, его адаптация решает почти всё.

2. Смена архитектуры даёт больше, чем лишняя разморозка. EfficientNet-B0 в том же режиме full на validation: 0,906 macro-F1 против 0,865 у ResNet18.

3. На test модель ведёт себя честно. Test использовали один раз, после выбора по validation. Получили accuracy 0,8974 и macro-F1 0,8970 — почти как на validation (0,907/0,906). Значит, модель не переобучилась и нормально обобщает.

4. Модель очень хрупка к гауссову шуму. Добавили гауссов шум σ=0,2 — accuracy упала с 0,90 до 0,33. При σ=0,4 — до 0,10, то есть случайное угадывание. Хуже того: 8 049 ранее верных предсказаний «сломались», и почти все стали говорить «airplane». Модель, когда ничего не видит, просто выдаёт один и тот же ответ — это называется «коллапс в константу».

5. К изменению яркости модель, наоборот, устойчива. Затемнение ×0,5 — accuracy 0,874, пересвет ×1,5 — 0,881. Падение максимум 2 п.п. Причина: глобальная яркость не ломает форму объекта, а нормализация частично её компенсирует. А вот локальный шум рвёт края и текстуры — это критично для предсказания.

6. Хуже всего распознаются пушистые классы: Cat — recall 0,727 (хуже всех), bird/deer/dog ~0,87. А техника — почти идеально: truck 0,952, automobile 0,949. Логично: на 32×32 кошка, собака и олень похожи, а у грузовика и корабля чёткая геометрия.

7. Огрнаичения запуска:  всего 8 000 train-картинок из 10 000, 5 эпох, один seed, CPU, вход 96 px вместо 224, только две архитектуры и два типа сдвига.


Главный вывод: Partial fine-tuning EfficientNet-B0 — лучший компромисс: ≈90% на чистых данных, но модель нужно защищать от локального шума; к глобальной яркости она устойчива.

